# Compare environments
The same checks side by side on every environment (your local stack, dev, stage): service versions, how many records each holds, and whether a search finds the same thing. Use it to tell a local problem from one that's on dev too.

Only reads, with plain public requests, so it's safe on every environment whatever this notebook is set to.

In [ ]:
import pandas as pd
import requests
from bluecore import environments, target

ENVS = {name: target(name) for name in environments()}
list(ENVS)

## Versions

In [ ]:
PATHS = {"API": "/api/health", "Workflows": "/workflows/health", "Sinopia": "/sinopia/health", "Marva": "/marva/health"}

def version(t, path):
    try:
        r = t.get(path, timeout=10)
        body = r.json() if r.ok else {}
        return body.get("version") or ("up" if r.ok else f"HTTP {r.status_code}")
    except (requests.RequestException, ValueError):
        return "unreachable"

pd.DataFrame({name: {svc: version(t, p) for svc, p in PATHS.items()} for name, t in ENVS.items()})

## How many records
Totals from search (hubs, works, instances) and the profile and other-resource lists.

In [ ]:
def total(t, path, **params):
    try:
        r = t.get(path, params={"limit": 1, **params}, timeout=30)
        return r.json().get("total") if r.ok else f"HTTP {r.status_code}"
    except (requests.RequestException, ValueError):
        return "unreachable"

pd.DataFrame({
    name: {
        "hubs": total(t, "/api/search/", q="", type="hubs"),
        "works": total(t, "/api/search/", q="", type="works"),
        "instances": total(t, "/api/search/", q="", type="instances"),
        "profiles": total(t, "/api/profiles/"),
        "other resources": total(t, "/api/resources/"),
    }
    for name, t in ENVS.items()
})

## The same search everywhere
Change `QUERY` to something you expect to find.

In [ ]:
QUERY = "pride and prejudice"

def top(t, q):
    try:
        r = t.get("/api/search/", params={"q": q, "limit": 3}, timeout=30).json()
        labels = [(x["type"], (x["data"].get("rdfs:label") or [x["data"].get("@id")])[0]) for x in r["results"]]
        return {"total": r["total"], "top results": labels}
    except (requests.RequestException, ValueError, KeyError):
        return {"total": "unreachable", "top results": []}

pd.DataFrame({name: top(t, QUERY) for name, t in ENVS.items()}).T